In [2]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("DataCleaning").getOrCreate()
df = spark.read.csv("customers_data.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()

+-----------+-------+----+-------+-------------+-------+--------------------+----------+----------+
|customer_id|   name| age| salary|      country|   city|               email|     phone| join_date|
+-----------+-------+----+-------+-------------+-------+--------------------+----------+----------+
|          1|  John | 150|-1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          1|  John |  25| 5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|
|          2|   mike|NULL|   NULL|United States|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|
|          4|   mike|  30|-1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|
|          5|  david|  45| 5000.0|        india|   NULL|                NULL|9876543210|      NULL|
+-----------+-------+----+-------+-------------+-------+--------------------+----------+----------+
only showing top 5 rows
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable

In [3]:
#find null counts

from pyspark.sql.functions import col, sum

null_counts = df.select([sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
null_counts.show()

+-----------+----+---+------+-------+----+-----+-----+---------+
|customer_id|name|age|salary|country|city|email|phone|join_date|
+-----------+----+---+------+-------+----+-----+-----+---------+
|          0| 159|125|   174|      0| 345|  260|  214|      199|
+-----------+----+---+------+-------+----+-----+-----+---------+



In [9]:
#Fill null values

df_fill = df.na.fill({
    "name": "Unknown",
    "city": "Unknown",
    "salary": 0
})

df_fill.show(20)

+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1|  John | 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1|  John |  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2|   mike|NULL|        0.0|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4|   mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|  david|  45|     5000.0|        india|Unknown|                NULL|    9876543210|      NULL|
|          6|  david|  45|        0.0|United States|Unknown|michael17@mcbride...|    9876543210|12-05-2024|
|          7|  SARAH|  25|  

In [16]:
# fill city using mode

from pyspark.sql.functions import count, desc

mode_row = (
	df.select("city")
	.where(col("city").isNotNull())
	.groupBy("city")
	.agg(count("*").alias("count"))
	.orderBy(desc("count"), col("city"))
	.first()
)

mode_city = mode_row["city"] if mode_row is not None else "Unknown"
df_fill = df.na.fill({"city": mode_city})
df_fill.show(20)

+-----------+-------+----+-----------+-------------+-------+--------------------+----------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|     phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+----------+----------+
|          1|  John | 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          1|  John |  25|     5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|
|          2|   Mike|NULL|       NULL|UNITED STATES|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|
|          4|   Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|
|          5|  David|  45|     5000.0|        INDIA|  Delhi|                NULL|9876543210|      NULL|
|          6|  David|  45|       NULL|UNITED STATES|  Delhi|michael17@mcbride...|9876543210|12-05-2024|
|          7|  Sarah|  25|    10000.0|          USA|  Delhi|    

In [10]:
#Convert all column names to lowercase and replace spaces with underscores.
df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])
df.show(20)

+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1|  John | 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1|  John |  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2|   mike|NULL|       NULL|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4|   mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|  david|  45|     5000.0|        india|   NULL|                NULL|    9876543210|      NULL|
|          6|  david|  45|       NULL|United States|   NULL|michael17@mcbride...|    9876543210|12-05-2024|
|          7|  SARAH|  25|  

In [8]:
# Trim leading and trailing spaces from name, city, and country.

from pyspark.sql.functions import trim

df = df.withColumn("name", trim(col("name"))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("country", trim(col("country")))
df.show(20)

+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id| name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-----+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1| John| 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1| John|  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2| mike|NULL|       NULL|United States|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4| mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|david|  45|     5000.0|        india|   NULL|                NULL|    9876543210|      NULL|
|          6|david|  45|       NULL|United States|   NULL|michael17@mcbride...|    9876543210|12-05-2024|
|          7|SARAH|  25|    10000.0|          

In [11]:
#Standardize customer names to Proper Case, country names to Upper Case, and city names to Proper Case.

from pyspark.sql.functions import initcap, upper

df = df.withColumn("name", initcap(col("name"))) \
    .withColumn("city", initcap(col("city"))) \
    .withColumn("country", upper(col("country")))
df.show(20)

+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1|  John | 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1|  John |  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2|   Mike|NULL|       NULL|UNITED STATES|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4|   Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|  David|  45|     5000.0|        INDIA|   NULL|                NULL|    9876543210|      NULL|
|          6|  David|  45|       NULL|UNITED STATES|   NULL|michael17@mcbride...|    9876543210|12-05-2024|
|          7|  Sarah|  25|  

In [12]:
#Convert empty strings in name, city, and email to NULL.

df = df.replace("", None, subset=["name", "city", "email"])
df.show(20)

+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|         phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+--------------+----------+
|          1|  John | 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|    9876543210|12-05-2024|
|          1|  John |  25|     5000.0|        U.S.A|  Delhi|                NULL|    9876543210|2024/01/01|
|          2|   Mike|NULL|       NULL|UNITED STATES|Chennai| sarah97@hotmail.com|          NULL|31-13-2024|
|          4|   Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|    9876543210|31-13-2024|
|          5|  David|  45|     5000.0|        INDIA|   NULL|                NULL|    9876543210|      NULL|
|          6|  David|  45|       NULL|UNITED STATES|   NULL|michael17@mcbride...|    9876543210|12-05-2024|
|          7|  Sarah|  25|  

In [14]:
#Remove special characters from phone numbers and keep only digits.

from pyspark.sql.functions import regexp_replace

df = df.withColumn("phone", regexp_replace(col("phone"), r"\D", ""))
df.show(20)

+-----------+-------+----+-----------+-------------+-------+--------------------+----------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|     phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+----------+----------+
|          1|  John | 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          1|  John |  25|     5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|
|          2|   Mike|NULL|       NULL|UNITED STATES|Chennai| sarah97@hotmail.com|      NULL|31-13-2024|
|          4|   Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|
|          5|  David|  45|     5000.0|        INDIA|   NULL|                NULL|9876543210|      NULL|
|          6|  David|  45|       NULL|UNITED STATES|   NULL|michael17@mcbride...|9876543210|12-05-2024|
|          7|  Sarah|  25|    10000.0|          USA|   NULL|    

In [39]:
#Separate records with invalid email IDs into a bad records dataframe and write them separately.

from pyspark.sql.functions import col

invalid_email_df = df.filter(~col("email").rlike(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"))
invalid_email_df.show(20)

+-----------+-------+----+-----------+-------------+-------+-------------+----------+----------+
|customer_id|   name| age|     salary|      country|   city|        email|     phone| join_date|
+-----------+-------+----+-----------+-------------+-------+-------------+----------+----------+
|          4|   Mike|  30|    -1000.0|          USA|Chennai|invalid_email|9876543210|31-13-2024|
|          7|  Sarah|  25|    10000.0|          USA|   NULL|invalid_email|      NULL|31-13-2024|
|         17|  David|  45|9.9999999E7|          USA|Chennai|invalid_email|     12345|2024/01/01|
|         18|  David|  30|    -1000.0|        INDIA|   NULL|invalid_email|9876543210|2024/01/01|
|         20|   NULL| 150|    -1000.0|        INDIA|   NULL|invalid_email|9876543210|2024/01/01|
|         23|   NULL|  30|     5000.0|          USA|   NULL|invalid_email|      NULL|      NULL|
|         26|   Mike|  30|       NULL|        INDIA|  Delhi|invalid_email|9876543210|31-13-2024|
|         28|   Mike|NULL|9.99

In [40]:
#Validate phone numbers and keep only 10-digit numbers. Write invalid phone records separately.

valid_phone_df = df.filter(col("phone").rlike(r"^\d{10}$"))
invalid_phone_df = df.filter(~col("phone").rlike(r"^\d{10}$"))
valid_phone_df.show(20)

+-----------+-------+----+-----------+-------------+-------+--------------------+----------+----------+
|customer_id|   name| age|     salary|      country|   city|               email|     phone| join_date|
+-----------+-------+----+-----------+-------------+-------+--------------------+----------+----------+
|          1|  John | 150|    -1000.0|        U.S.A|  Delhi|floresrandall@liv...|9876543210|12-05-2024|
|          1|  John |  25|     5000.0|        U.S.A|  Delhi|                NULL|9876543210|2024/01/01|
|          4|   Mike|  30|    -1000.0|          USA|Chennai|       invalid_email|9876543210|31-13-2024|
|          5|  David|  45|     5000.0|        INDIA|   NULL|                NULL|9876543210|      NULL|
|          6|  David|  45|       NULL|UNITED STATES|   NULL|michael17@mcbride...|9876543210|12-05-2024|
|          8|  Sarah|  45|    10000.0|        INDIA| Mumbai|                NULL|9876543210|2024/01/01|
|         10|   NULL| 150|9.9999999E7|        U.S.A| Mumbai|lero

In [ ]:
#Convert age column into integer datatype and handle invalid string values and nulls.

from pyspark.sql.functions import expr

df = df.withColumn(
    "age",
    expr["try age")
df.show(20)

{"ts": "2026-05-08 10:51:08.512", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "line 5 in cell [42]", "line": "", "fragment": "cast", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o618.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value 'abc' of the type \"STRING\" cannot be cast to \"INT\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"cast\" was called from\nline 5 in cell [42]\n\r\n\tat org.apache.spark.sql.errors.QueryExecutionErrors

NumberFormatException: [CAST_INVALID_INPUT] The value 'abc' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 5 in cell [42]
